# Customer Churn Prediction — Model Experiments

This revised notebook keeps the test set untouched until one final evaluation.

> **Source of truth:** `src/train.py` is the canonical reusable training pipeline for this project. This notebook mirrors the same methodology for transparent experimentation, inspection, and portfolio presentation. If the notebook and script ever diverge, `src/train.py` should be treated as authoritative.

The notebook saves its model artifact under a separate filename so running it does not overwrite the production-style artifact created by `src/train.py`.


## 1. Imports

In [1]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_val_predict, cross_validate, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

## 2. Load and Clean the Data

`TotalCharges` is numeric in meaning but can contain blank strings. Invalid values are converted to `NaN`, and customers with `tenure == 0` are assigned `TotalCharges = 0`.

In [2]:
DATA_PATH = Path("../data/raw/Telco-Customer-Churn.csv")
df = pd.read_csv(DATA_PATH)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.loc[df["tenure"] == 0, "TotalCharges"] = 0
print("Shape:", df.shape)
print("Missing values after cleaning:", df.isna().sum().sum())
df.head()

Shape: (7043, 21)
Missing values after cleaning: 0


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Prepare Features and Target

`customerID` is removed because it is an identifier. Target encoding: `No → 0`, `Yes → 1`.

In [3]:
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({"No": 0, "Yes": 1})
print("Feature shape:", X.shape)
print(y.value_counts(normalize=True))

Feature shape: (7043, 19)
Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64


## 4. Train-Test Split

The split is stratified to preserve class balance. **The test set remains untouched until the final evaluation.**

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True))
print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

Training shape: (5634, 19)
Test shape: (1409, 19)

Training class distribution:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

Test class distribution:
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


## 5. Define Feature Groups

`SeniorCitizen` is treated as categorical because it is a binary flag rather than a continuous measurement.

In [5]:
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [c for c in X_train.columns if c not in numeric_features]
print("Numerical:", numeric_features)
print("Categorical:", categorical_features)

Numerical: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 6. Preprocessing Pipeline

Numeric: median imputation + standardization. Categorical: most-frequent imputation + one-hot encoding. All preprocessing stays inside the sklearn pipeline to avoid leakage.

In [6]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

## 7. Dummy Baseline

The baseline is evaluated with 5-fold stratified cross-validation on the training set only.

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"accuracy":"accuracy","precision":"precision","recall":"recall","f1":"f1","roc_auc":"roc_auc"}
dummy_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])
dummy_scores = cross_validate(dummy_pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
pd.DataFrame({
    "Metric": scoring.keys(),
    "Mean": [dummy_scores[f"test_{m}"].mean() for m in scoring]
})

,Metric,Mean
0,accuracy,0.734647
1,precision,0.000000
2,recall,0.000000
3,f1,0.000000
4,roc_auc,0.500000


## 8. Candidate Model Comparison

This is a screening comparison on training data only. Final selection happens after tuning.

In [8]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}
results=[]
for name, model in models.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor),("model", model)])
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results.append({
        "Model":name,
        "Accuracy Mean":scores["test_accuracy"].mean(),
        "Precision Mean":scores["test_precision"].mean(),
        "Recall Mean":scores["test_recall"].mean(),
        "F1 Mean":scores["test_f1"].mean(),
        "F1 Std":scores["test_f1"].std(),
        "ROC-AUC Mean":scores["test_roc_auc"].mean(),
        "ROC-AUC Std":scores["test_roc_auc"].std()
    })
results_df = pd.DataFrame(results).sort_values("F1 Mean", ascending=False)
results_df

,Model,Accuracy Mean,Precision Mean,Recall Mean,F1 Mean,F1 Std,ROC-AUC Mean,ROC-AUC Std
0,Logistic Regression,0.801918,0.652094,0.543813,0.592412,0.029782,0.846149,0.012562
3,Gradient Boosting,0.803340,0.662221,0.529097,0.587972,0.029636,0.848200,0.012550
2,Random Forest,0.786655,0.627344,0.485619,0.547080,0.020813,0.820246,0.012123
1,Decision Tree,0.732161,0.495745,0.500334,0.497541,0.013957,0.658235,0.009795


## 9. Hyperparameter Tuning — Logistic Regression

F1 is the primary tuning metric because no real business cost matrix is available.

In [9]:
logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])
logistic_param_grid = {
    "model__C":[0.01,0.1,1,10,100],
    "model__class_weight":[None,"balanced"]
}
logistic_grid = GridSearchCV(logistic_pipeline, logistic_param_grid, scoring="f1", cv=cv, n_jobs=-1, refit=True)
logistic_grid.fit(X_train, y_train)
print("Best parameters:", logistic_grid.best_params_)
print("Best CV F1:", logistic_grid.best_score_)

Best parameters: {'model__C': 0.1, 'model__class_weight': 'balanced'}
Best CV F1: 0.6287682721581708


## 10. Hyperparameter Tuning — Gradient Boosting

In [10]:
gb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])
gb_param_grid = {
    "model__n_estimators":[100,200],
    "model__learning_rate":[0.05,0.1],
    "model__max_depth":[2,3]
}
gb_grid = GridSearchCV(gb_pipeline, gb_param_grid, scoring="f1", cv=cv, n_jobs=-1, refit=True)
gb_grid.fit(X_train, y_train)
print("Best parameters:", gb_grid.best_params_)
print("Best CV F1:", gb_grid.best_score_)

Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}
Best CV F1: 0.5879715423174938


## 11. Final Model Selection

The final candidate is selected by tuned cross-validated F1. Logistic Regression also offers simpler interpretation when performance is stronger.

In [11]:
tuned_model_results = pd.DataFrame([
    {"Model":"Logistic Regression","Best CV F1":logistic_grid.best_score_},
    {"Model":"Gradient Boosting","Best CV F1":gb_grid.best_score_}
]).sort_values("Best CV F1", ascending=False)
tuned_model_results

,Model,Best CV F1
0,Logistic Regression,0.628768
1,Gradient Boosting,0.587972


In [12]:
if logistic_grid.best_score_ >= gb_grid.best_score_:
    selected_model_name = "Logistic Regression"
    selected_model = clone(logistic_grid.best_estimator_)
else:
    selected_model_name = "Gradient Boosting"
    selected_model = clone(gb_grid.best_estimator_)
print("Selected model:", selected_model_name)

Selected model: Logistic Regression


## 12. Threshold Selection with Out-of-Fold Probabilities

Threshold selection uses OOF probabilities from `X_train` only. Each row is predicted by a model that did not train on that row.

In [13]:
oof_proba = cross_val_predict(
    selected_model, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1
)[:,1]
thresholds = np.round(np.arange(0.20,0.81,0.01),2)
threshold_results=[]
for threshold in thresholds:
    preds=(oof_proba>=threshold).astype(int)
    threshold_results.append({
        "Threshold":threshold,
        "Accuracy":accuracy_score(y_train,preds),
        "Precision":precision_score(y_train,preds,zero_division=0),
        "Recall":recall_score(y_train,preds),
        "F1":f1_score(y_train,preds)
    })
threshold_results_df=pd.DataFrame(threshold_results)
threshold_results_df.sort_values("F1",ascending=False).head(10)

,Threshold,Accuracy,Precision,Recall,F1
34,0.54,0.766596,0.542214,0.773244,0.637441
35,0.55,0.768726,0.545845,0.764548,0.636946
40,0.60,0.784345,0.575676,0.712375,0.636771
37,0.57,0.774050,0.555334,0.745151,0.636390
36,0.56,0.771033,0.550073,0.753177,0.635799
33,0.53,0.762513,0.536192,0.777926,0.634825
39,0.59,0.779553,0.566544,0.720401,0.634276
41,0.61,0.785233,0.578599,0.701672,0.634220
32,0.52,0.759141,0.531250,0.784615,0.633540
38,0.58,0.775825,0.559487,0.729766,0.633382


In [14]:
best_row = threshold_results_df.loc[threshold_results_df["F1"].idxmax()]
best_threshold = float(best_row["Threshold"])
print("Selected threshold:", best_threshold)
print(best_row)

Selected threshold: 0.54
Threshold    0.540000
Accuracy     0.766596
Precision    0.542214
Recall       0.773244
F1           0.637441
Name: 34, dtype: float64


### Threshold Selection Note

This threshold is selected by OOF F1 because the project does not have a real retention-offer cost matrix. In production, threshold selection should use real business costs.

## 13. Fit Final Model on All Training Data

In [15]:
final_model = clone(selected_model)
final_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='pa

## 14. Final Evaluation on the Untouched Test Set

This is the first model evaluation on `X_test`. No tuning decisions should be made after seeing these results.

In [16]:
test_proba = final_model.predict_proba(X_test)[:,1]
final_pred = (test_proba >= best_threshold).astype(int)
print("Confusion Matrix:")
print(confusion_matrix(y_test, final_pred))
print("\nClassification Report:")
print(classification_report(y_test, final_pred, zero_division=0))
final_metrics = {
    "Accuracy":accuracy_score(y_test,final_pred),
    "Precision":precision_score(y_test,final_pred,zero_division=0),
    "Recall":recall_score(y_test,final_pred),
    "F1":f1_score(y_test,final_pred),
    "ROC-AUC":roc_auc_score(y_test,test_proba),
    "PR-AUC":average_precision_score(y_test,test_proba)
}
pd.Series(final_metrics,name="Test Score")

Confusion Matrix:
[[780 255]
 [ 91 283]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.75      0.82      1035
           1       0.53      0.76      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.77      1409



Accuracy     0.754436
Precision    0.526022
Recall       0.756684
F1           0.620614
ROC-AUC      0.840830
PR-AUC       0.633034
Name: Test Score, dtype: float64

## 15. Train vs Test Diagnostic

Training metrics are shown only as an overfitting diagnostic, not for further tuning.

In [17]:
train_proba = final_model.predict_proba(X_train)[:,1]
train_pred=(train_proba>=best_threshold).astype(int)
train_metrics={
    "Accuracy":accuracy_score(y_train,train_pred),
    "Precision":precision_score(y_train,train_pred,zero_division=0),
    "Recall":recall_score(y_train,train_pred),
    "F1":f1_score(y_train,train_pred),
    "ROC-AUC":roc_auc_score(y_train,train_proba),
    "PR-AUC":average_precision_score(y_train,train_proba)
}
pd.DataFrame({"Train":train_metrics,"Test":final_metrics})

,Train,Test
Accuracy,0.767661,0.754436
Precision,0.543458,0.526022
Recall,0.777926,0.756684
F1,0.639890,0.620614
ROC-AUC,0.848530,0.840830
PR-AUC,0.664039,0.633034


### Overfitting Check

The training and test metrics are relatively close across all major evaluation measures.

This suggests that the final Logistic Regression model generalizes reasonably well and does not show strong signs of overfitting.

Small performance differences between training and test sets are expected.

## 16. Model Interpretation

If Logistic Regression is selected, coefficients are regularized associations with model log-odds, not causal effects or clean reference-category contrasts. Numeric coefficients correspond to standardized features. Odds ratios are included for readability.

In [18]:
if selected_model_name == "Logistic Regression":
    feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
    logistic_model = final_model.named_steps["model"]
    coefficients = logistic_model.coef_[0]
    feature_importance = pd.DataFrame({
        "Feature":feature_names,
        "Coefficient":coefficients,
        "OddsRatio":np.exp(coefficients)
    }).sort_values("Coefficient",ascending=False)
    print("Top positive coefficients:")
    display(feature_importance.head(10))
    print("\nTop negative coefficients:")
    display(feature_importance.tail(10))
else:
    print("Coefficient interpretation skipped because the selected model is not Logistic Regression.")

Top positive coefficients:


,Feature,Coefficient,OddsRatio
37,cat__Contract_Month-to-month,0.684518,1.982815
17,cat__InternetService_Fiber optic,0.399905,1.491684
2,num__TotalCharges,0.284917,1.329652
44,cat__PaymentMethod_Electronic check,0.283824,1.328199
19,cat__OnlineSecurity_No,0.253625,1.288688
28,cat__TechSupport_No,0.220702,1.246951
41,cat__PaperlessBilling_Yes,0.165699,1.180218
36,cat__StreamingMovies_Yes,0.151540,1.163625
33,cat__StreamingTV_Yes,0.140219,1.150525
22,cat__OnlineBackup_No,0.121366,1.129038



Top negative coefficients:


,Feature,Coefficient,OddsRatio
10,cat__Dependents_Yes,-0.116483,0.890046
42,cat__PaymentMethod_Bank transfer (automatic),-0.117337,0.889285
43,cat__PaymentMethod_Credit card (automatic),-0.119937,0.886976
30,cat__TechSupport_Yes,-0.142626,0.867078
40,cat__PaperlessBilling_No,-0.168428,0.844992
13,cat__MultipleLines_No,-0.171734,0.842203
21,cat__OnlineSecurity_Yes,-0.175549,0.838996
16,cat__InternetService_DSL,-0.321830,0.724821
39,cat__Contract_Two year,-0.651307,0.521364
0,num__tenure,-0.971786,0.378406


## 17. Error Analysis

This is post-evaluation analysis to understand limitations. These findings must not be used to retune the model on the same test set.

In [19]:
error_analysis=X_test.copy()
error_analysis["Actual"]=y_test
error_analysis["Predicted"]=final_pred
error_analysis["ChurnProbability"]=test_proba
false_negatives=error_analysis[(error_analysis["Actual"]==1)&(error_analysis["Predicted"]==0)]
false_positives=error_analysis[(error_analysis["Actual"]==0)&(error_analysis["Predicted"]==1)]
actual_churn=error_analysis[error_analysis["Actual"]==1]
actual_no_churn=error_analysis[error_analysis["Actual"]==0]
print("False negatives:",len(false_negatives))
print("False positives:",len(false_positives))
print("Actual churn:",len(actual_churn))
print("Actual no churn:",len(actual_no_churn))

False negatives: 91
False positives: 255
Actual churn: 374
Actual no churn: 1035


In [20]:
print("False Negative numeric summary:")
display(false_negatives[["tenure","MonthlyCharges","TotalCharges","ChurnProbability"]].describe())
print("\nFalse Positive numeric summary:")
display(false_positives[["tenure","MonthlyCharges","TotalCharges","ChurnProbability"]].describe())

False Negative numeric summary:


,tenure,MonthlyCharges,TotalCharges,ChurnProbability
count,91.000000,91.00000,91.000000,91.000000
mean,28.241758,60.19011,2330.373077,0.356162
std,24.424187,32.04307,2530.121072,0.129083
min,1.000000,18.95000,19.300000,0.019044
25%,4.000000,20.80000,129.925000,0.264108
50%,24.000000,55.70000,1498.200000,0.378886
75%,51.000000,93.27500,3987.750000,0.459893
max,72.000000,115.65000,7968.850000,0.532115



False Positive numeric summary:


,tenure,MonthlyCharges,TotalCharges,ChurnProbability
count,255.000000,255.000000,255.000000,255.000000
mean,19.078431,77.217255,1645.281765,0.718537
std,16.162369,20.435241,1618.698543,0.096703
min,1.000000,24.200000,24.200000,0.548049
25%,6.000000,68.600000,403.500000,0.647620
50%,16.000000,79.950000,1043.400000,0.711825
75%,27.000000,92.950000,2377.950000,0.798226
max,67.000000,114.100000,7132.150000,0.918956


### Compare Error Groups with Reference Populations

In [21]:
def compare_category(error_group, reference_group, feature):
    error_dist=error_group[feature].value_counts(normalize=True).rename("ErrorGroup")
    ref_dist=reference_group[feature].value_counts(normalize=True).rename("ReferenceGroup")
    comparison=pd.concat([error_dist,ref_dist],axis=1).fillna(0)
    comparison["Difference"]=comparison["ErrorGroup"]-comparison["ReferenceGroup"]
    return comparison.sort_values("Difference",ascending=False)

In [22]:
for feature in ["Contract","InternetService","TechSupport"]:
    print(f"\nFALSE NEGATIVES — {feature}")
    display(compare_category(false_negatives,actual_churn,feature))
    print(f"FALSE POSITIVES — {feature}")
    display(compare_category(false_positives,actual_no_churn,feature))


FALSE NEGATIVES — Contract


,ErrorGroup,ReferenceGroup,Difference
Contract,,,
One year,0.329670,0.096257,0.233414
Two year,0.098901,0.024064,0.074837
Month-to-month,0.571429,0.879679,-0.308251


FALSE POSITIVES — Contract


,ErrorGroup,ReferenceGroup,Difference
Contract,,,
Month-to-month,0.956863,0.428986,0.527877
One year,0.043137,0.255072,-0.211935
Two year,0.000000,0.315942,-0.315942



FALSE NEGATIVES — InternetService


...

,ErrorGroup,ReferenceGroup,Difference
InternetService,,,
Fiber optic,0.72549,0.348792,0.376698
DSL,0.27451,0.373913,-0.099403
No,0.00000,0.277295,-0.277295



FALSE NEGATIVES — TechSupport


,ErrorGroup,ReferenceGroup,Difference
TechSupport,,,
No internet service,0.274725,0.066845,0.207880
Yes,0.351648,0.160428,0.191221
No,0.373626,0.772727,-0.399101


FALSE POSITIVES — TechSupport


,ErrorGroup,ReferenceGroup,Difference
TechSupport,,,
No,0.792157,0.399034,0.393123
Yes,0.207843,0.323671,-0.115828
No internet service,0.000000,0.277295,-0.277295


## 18. Error Analysis Findings

The final model produced:

- 91 false negatives
- 255 false positives

### False Negatives

False negatives were disproportionately associated with customer profiles that are usually linked with lower churn risk.

Compared with all actual churn customers:

- One-year and two-year contracts were overrepresented.
- DSL and no-internet customers were overrepresented.
- Customers with technical support were overrepresented.
- Month-to-month and fiber-optic customers were underrepresented.

This suggests that the model has more difficulty identifying churners whose profiles do not match the dominant high-risk churn patterns learned from the training data.

### False Positives

False positives were strongly concentrated among customers with commonly observed high-risk characteristics.

Compared with all actual non-churn customers:

- Month-to-month contracts were strongly overrepresented.
- Fiber-optic internet was strongly overrepresented.
- Customers without technical support were strongly overrepresented.

This suggests that the model may over-predict churn for customers who resemble the typical high-risk churn profile but ultimately remain with the company.

These findings describe model behavior and statistical associations only. They do not establish causal relationships and are not used to retune the model on the test set.


## 19. Save the Notebook Model Bundle

For reproducibility, the notebook saves its fitted model and selected threshold together.

This notebook intentionally writes to `models/notebook_churn_model.joblib` rather than `models/churn_model.joblib`. The canonical artifact used by the reusable application workflow is produced by `src/train.py`, so running this notebook will not overwrite it.


In [23]:
MODEL_PATH = Path("../models/notebook_churn_model.joblib")
model_bundle={"model":final_model,"threshold":best_threshold}
joblib.dump(model_bundle,MODEL_PATH)
print("Saved model to:",MODEL_PATH)
print("Saved threshold:",best_threshold)


Saved model to: ..\models\churn_model.joblib
Saved threshold: 0.54


## 20. Reload Smoke Test

A manually defined customer is used instead of reusing a test-set row.

In [24]:
loaded_bundle=joblib.load(MODEL_PATH)
loaded_model=loaded_bundle["model"]
loaded_threshold=loaded_bundle["threshold"]
sample_customer={
    "gender":"Male","SeniorCitizen":0,"Partner":"No","Dependents":"No","tenure":2,
    "PhoneService":"Yes","MultipleLines":"No","InternetService":"Fiber optic",
    "OnlineSecurity":"No","OnlineBackup":"No","DeviceProtection":"No","TechSupport":"No",
    "StreamingTV":"Yes","StreamingMovies":"Yes","Contract":"Month-to-month",
    "PaperlessBilling":"Yes","PaymentMethod":"Electronic check",
    "MonthlyCharges":95.0,"TotalCharges":190.0
}
sample_df=pd.DataFrame([sample_customer])
sample_probability=loaded_model.predict_proba(sample_df)[:,1][0]
sample_prediction=int(sample_probability>=loaded_threshold)
print("Loaded threshold:",loaded_threshold)
print("Sample churn probability:",sample_probability)
print("Sample prediction:",sample_prediction)

Loaded threshold: 0.54
Sample churn probability: 0.9104767034693579
Sample prediction: 1


## 21. Final Methodology Summary

- Preprocessing is fitted inside pipelines.
- Model comparison uses training data only.
- Hyperparameter tuning uses cross-validation on training data only.
- Threshold selection uses OOF training probabilities only.
- The test set is evaluated once after all choices are complete.
- Error analysis is post-evaluation and not used for retuning.
- Interpretation is associative, not causal.